<h1 style="text-align:center;">Quantifying Acne Severity Dynamics - A Hierarchical Bayesian Approach</h1>
<p style="text-align:center;">By Nathaniel Wolff.</p>



# Analysis Structure and Results

Dataset is found here: https://www.kaggle.com/datasets/manuelhettich/acne04. Consists of n = 10 synthetic patients, 
with acne severity recorded over 112 distinct days of treatment (each indexing a distinct treatment history).


### Analysis Steps
#### 1) Data Parsing -
Base dataframe is seperated by patient. Raw acne severities are normalized as % change relative to patient average baseline intensity.

#### 2) Treatment History Metadata Addition - 
A treatment history metadata column is added to each patient dataframe. Histories are of the form ( (Treatment $a_{1}$, Day 1),...(Treatment $a_{n}$, Day i)), where n is the index of a given treatment in the full history.

# Model Structure and Parameter Estimation


### Hieararchical Bayesian Model Structure

Here, Acne Severity is modeled as a random variable $S_{t}$, distributed according to the following Bayesian Hieararchial Model, smoothed with a Kalman Filter. 

#### Layer 1: 
The lowest layer of the hierarchy denotes the effect of a treatment vector, $t = \begin{pmatrix} Days_{Antibiotics} & Days_{Retinol} \end{pmatrix}$, on the distribution of clinically practical diagnostics, $\hat{o} = \begin{bmatrix} pH_{skin} & [Insulin] & [IGF1] & NLR\end{bmatrix}$. These raw diagnostic biomarkers are embedded in a feature space, defined by the following features modeling important contributors to acne pathology, per biochemical and immunological mechanisms:

Bacterial Dysbiosis Lipase Effect, $L_{t}$: $\sum_{i=1}^{n} \frac{1}{\sigma_{i}\sqrt{2\pi}} \exp\left(-\frac{(pH_{skin} - \mu_{i})^2)}{2\sigma^2_{i}}\right)$ (to be later updated with kinetic modeling approaches). 

mTORC1 Effect, $MT_{T}$: $\alpha_{1} * [Insulin]/([Insulin] + (K_{d}))_{Insulin} + \alpha_{2} * [IFG1]/([IFG1] + (K_{d}))_{IFG1}$. 

Partial Inflammatory Effect, $I_{t}$: $\alpha_{3} * NLR$

#### Layer 2: 
This vector of these effects, $\hat{o'}$, is mapped to a vector of imputed biomarkers via the linear transformation W, and serves as the mean for the distribution of imputed biomarkers, $\hat{Y_{t}}$ = $\begin{bmatrix} Lipase\,Effect & mTORC1 & Inflammam\end{bmatrix}$, which is distributed normallly: $\hat{Y_{t}} \sim \mathcal{N}(W\hat{o'},\,\Sigma_{1})\,$. 

#### Layer 3: 
In turn, $\hat{Y_{t}}$ is mapped to a full latent state vector, $\hat{Z_{t}}$, defined the same way as the data-driven approach: $\hat{Z_{t}} = \begin{bmatrix} B_{t} \\ I_{t} \\ S_{t} \end{bmatrix}$. The mapping, via linear transformation A, defines the latent state's own normal distribution: $\hat{Z_{t}} \sim \mathcal{N}(A\hat{Y_{t}} + OU,\,\Sigma_{2})\,$, where $OU$ is a stochastic contribution from the Ornstein-Uhlenbeck process governing the reversion of the latent states to the respective mean, as a result of other biological pathways not explicitly modeled here.

#### Layer 4:
The final layer determines the distribution of acne severity, $S_{t}$, as a function of $\hat{Y_{t}}$. Here, acne severity is imputed to follow a Normal - Gamma distribution, such that the maximized likelihood from the first 3 layers is conjugate to the Gamma prior. As such,

$S_{t} \sim \Gamma(\alpha_{n}, \beta_{n})$ where $ \alpha_{n} = $



### Expectation-Maximization of Model Parameters and Latent State Acne Severity Change State Distribution
 
This implementation applies EM to the joint likelihood of all layers, assuming the Markov propery holds between each. That is, the likelihood maximized is: $$\mathcal{L} = \ln Pr(Latent,\,Imputed,\,Observed)  = \ln Pr(L | I, \theta_{2}) + \ln Pr(I | O, \theta_{1}) + Const.$$

The expectation step calculates the first moments for both Y and Z, $\mathcal{E}(\hat y_t), \mathcal{E}(\hat z_t)$, as well as the second moment matrices, $\mathcal{E}(\hat y_t \hat y_t^{T}), \mathcal{E}(\hat z_t \hat z_t^{T})$, with posterior variances included. 

The cross moments between Y and Z, $\mathcal{E}(\hat z_t \hat y_t^{T})$ and $\mathcal{E}(\hat z_{t} \hat z_{t-1}^{T})$ are also found here. 



$$\textbf{Maximization of Parameters}$$

The maximization step maximizes both the weights/biases and the model parameters, that is, $\theta_{1}, \theta_{2}$, W, and A.  

In [ ]:
from acne_model import data_parsing
from acne_model import data_visualization
from acne_model import model_building, process_data_and_build_HBM
import json
import numpy as np

def main(this_raw_data_name, json_name):
    data_returns = data_parsing(this_raw_data_name)
    

    these_frames = data_returns[1]
    these_averages = data_returns[5]
    
    these_dirichlets = data_returns[6]
    these_empirical_trans_matrices = data_returns[7]
    transition_counts = data_returns[8]
    these_raw_frames = data_returns[9]

    view_plots = False
    user_input = input("Do you want to view the data? Type Yes or No.")
    
    if user_input == "Yes":
         view_plots = True
    if view_plots:
        this_data_visualization = data_visualization(these_frames, data_returns[0], data_returns[4], data_returns[3])
    if not view_plots:
        print("Ok, going on to modeling.")
        this_HBM = process_data_and_build_HBM(these_frames, model_priors_and_config_handle = json_name, these_raw_frames = these_raw_frames)
    
#retrieving Acne04 dataset from Kaggle source (local download on machine)
this_raw_data_name = "sim_acne_amended_v2.csv"
this_json_name = "bhm_model_config.json"
if __name__ == "__main__":
    main(this_raw_data_name, this_json_name)

## 